# Chapter 8 — Bayesian Networks: Drawing What You Already Know

Companion notebook for the [probintro textbook](https://josephausterweil.github.io/probintro/intro2/08_bayes_nets/).

You'll build three Bayes nets in GenJAX — the mystery-bento mixture as a graph, a hierarchical version, and Chibany's multi-parent bento network — then run **inference** backward from an observed weight to a hidden cluster.


## Setup

Run this first. On Colab you may need to restart the runtime afterward (Runtime → Restart runtime).


In [ ]:
!pip install genjax -q
print('✅ ready')

## 1. The mixture model, drawn as a Bayes net ($z \to x$)

One cluster node `z` flows into one weight node `x`. Running it forward is **ancestral sampling**.


In [ ]:
import jax
import jax.numpy as jnp
import jax.random as jr
from genjax import gen, flip, normal

MU_LIGHT = 350.0   # hamburger cluster
MU_HEAVY = 500.0   # tonkatsu cluster
SIGMA = 40.0       # wide enough that the clusters overlap

@gen
def one_bento():
    z = flip(0.5) @ "z"
    mu = jnp.where(z, MU_HEAVY, MU_LIGHT)
    x = normal(mu, SIGMA) @ "x"
    return z, x

z, x = one_bento.simulate(jr.key(0), ()).get_retval()
label = "tonkatsu (heavy)" if bool(z) else "hamburger (light)"
print(f"Sampled cluster: {label}")
print(f"Sampled weight:  {float(x):.1f} g")

## 2. Adding a hyperprior ($\alpha \to \pi \to z \to x$)

Now the mixing weight `pi` is itself random, drawn from a Beta prior.


In [ ]:
from genjax import beta

@gen
def one_bento_hierarchical():
    pi = beta(2.0, 2.0) @ "pi"
    z = flip(pi) @ "z"
    mu = jnp.where(z, MU_HEAVY, MU_LIGHT)
    x = normal(mu, SIGMA) @ "x"
    return pi, z, x

pi, z, x = one_bento_hierarchical.simulate(jr.key(1), ()).get_retval()
print(f"Sampled mixing weight pi: {float(pi):.2f}")
print(f"Sampled weight:           {float(x):.1f} g")

## 3. Chibany's multi-parent network ($W, D, R \to B$)

Three independent causes feed one bento via an 8-entry conditional probability table.


In [ ]:
@gen
def chibany_bento_network():
    weather = flip(0.5) @ "weather"        # True = hot
    day = flip(0.6) @ "day"                # True = late-week
    restaurant = flip(0.7) @ "restaurant"  # True = cafeteria B

    cpt = jnp.array([
        [[0.5, 0.7], [0.7, 0.9]],   # weather = cold
        [[0.3, 0.5], [0.5, 0.7]],   # weather = hot
    ])
    p_tonkatsu = cpt[weather.astype(int), day.astype(int), restaurant.astype(int)]
    bento = flip(p_tonkatsu) @ "bento"
    return bento

keys = jr.split(jr.key(2), 20000)
bentos = jax.vmap(lambda k: chibany_bento_network.simulate(k, ()).get_retval())(keys)
print(f"P(bento = tonkatsu) ≈ {float(jnp.mean(bentos.astype(float))):.3f}")

## 4. Inference: from effect back to cause

Condition on an observed weight and infer the hidden cluster — running the net *against* its arrows.


In [ ]:
from genjax import ChoiceMap

observation = ChoiceMap.d({"x": 430.0})

keys = jr.split(jr.key(3), 8000)
def infer_cluster(k):
    trace, log_weight = one_bento.generate(k, observation, ())
    return trace.get_choices()["z"].astype(float), log_weight

z_samples, log_weights = jax.vmap(infer_cluster)(keys)
weights = jnp.exp(log_weights - jnp.max(log_weights))
weights = weights / jnp.sum(weights)
p_heavy = jnp.sum(z_samples * weights)
print(f"P(cluster = heavy | weight = 430 g) ≈ {float(p_heavy):.3f}")

## Your turn

1. Add a holiday cause `H` to `chibany_bento_network` and re-estimate the tonkatsu marginal.
2. Estimate $P(B = \text{tonkatsu} \mid \text{weather} = \text{hot})$ by keeping only the hot-weather samples.
3. Try different observed weights in cell 4 (360 g, 500 g) and watch the posterior over the cluster move.
